<a href="https://colab.research.google.com/github/danishsyed-dev/Basic-Python-Network/blob/main/E_Commerces_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Building Beginner ML Model**

In [115]:
df = pd.read_csv('https://raw.githubusercontent.com/mar-antaya/ml-portfolio-course/main/ecommerce_data.csv')
print(df.shape)

(8205, 7)


In [116]:
df.head()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8205 entries, 0 to 8204
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       8189 non-null   object 
 1   InvoiceDate      8205 non-null   object 
 2   Quantity         8205 non-null   int64  
 3   UnitPrice        8205 non-null   float64
 4   TotalAmount      8205 non-null   float64
 5   Country          6866 non-null   object 
 6   ProductCategory  8205 non-null   object 
dtypes: float64(2), int64(1), object(4)
memory usage: 448.8+ KB


,Quantity,UnitPrice,TotalAmount
count,8205.000000,8205.000000,8205.000000
mean,1.248507,181.370449,212.411539
std,0.792325,199.572215,255.546153
min,-5.000000,0.000000,4.580000
25%,1.000000,38.980000,44.260000
50%,1.000000,98.300000,117.220000
75%,1.000000,250.500000,280.430000
max,8.000000,898.540000,2751.720000


**Step 1: Analyzing the Data**
Now that we have the summary of our dataset, we need to understand what we are working with before we start cleaning it.

**1. What are we doing?** We are performing an initial data audit. This means looking at the columns, their types, and checking for common issues like missing values or strange numbers (outliers) that could confuse a Machine Learning model later.

**2. The Analysis of the Output:**


*   **Problem Type:** This is a Classification problem. We want to predict which `ProductCategory` a transaction belongs to based on numerical values like price and quantity.
*   **Target Column:** The target is `ProductCategory`. This is the 'label' we want the AI to learn to guess.
*   **Features:** We will likely use `Quantity`, `UnitPrice`, and `TotalAmount` as our features (inputs).
*   **Data Problems Spotted:**

    *   **Missing Values:** *`CustomerID`* and *`Country`* have missing rows (only 6866 and 8189 non-null values out of 8205).
    *   **Wrong Data Types:** `InvoiceDate` is an 'object' (text), but it should be a 'datetime' format if we want to use time in our model.
    *   **Outliers/Errors:** Look at `describe()`: the minimum `Quantity` is -5. You cannot have a negative quantity in a normal sale! This likely represents a return or a data error.
**3. Your Next Decision:** Based on these findings, our next step is Data Cleaning. We need to remove those negative quantities and handle the missing data so the model has a clean 'textbook' to learn from.

**4. What to type next:** Copy and run the code below to clean these specific issues.



In [117]:
# 1. Remove rows where Quantity is 0 or less
df_cleaned = df[df['Quantity'] > 0].copy()

# 2. Fill missing Country values with 'Unknown'
df_cleaned['Country'] = df_cleaned['Country'].fillna('Unknown')

# 3. Drop rows where CustomerID is missing since we can't track that specific user
df_cleaned = df_cleaned.dropna(subset=['CustomerID'])

print(f"New data shape: {df_cleaned.shape}")
display(df_cleaned.describe())

New data shape: (8124, 7)


,Quantity,UnitPrice,TotalAmount
count,8124.000000,8124.000000,8124.000000
mean,1.268464,181.523949,212.721311
std,0.760780,199.824608,256.150546
min,1.000000,0.000000,4.580000
25%,1.000000,39.120000,44.375000
50%,1.000000,98.360000,117.220000
75%,1.000000,250.047500,280.500000
max,8.000000,898.540000,2751.720000


**Step 2: Preparing Features and Target**

Now that our data is clean, we need to pick which parts of the data the AI will use to learn, and what exactly it is trying to predict.

1. What are we doing? We are splitting our data into Features (X) and a Target (y).


    *   **Features (X):** The inputs (like Quantity and Price) that help the model make a guess.

    *   **Target (y):** The answer key (ProductCategory) we want the model to learn.

    *   **Encoding:** Computers don't understand words like 'Books' or 'Clothing', so we use a `LabelEncoder` to turn those categories into numbers (0, 1, 2, etc.).

**2. The Analysis of the Output:** You will see a list of the original category names and their new numerical codes.

**3. Your Next Decision:** Once the numbers are ready, we will split the data into a 'study set' (training) and a 'test set' to see how well the model learned.

**4. What to type next:** Run this code to define your features and transform your target into numbers.



In [118]:
from sklearn.preprocessing import LabelEncoder

# 1. Select the features (inputs) we want to use
# We use the cleaned data from the previous step
X = df_cleaned[['Quantity', 'UnitPrice', 'TotalAmount']]

# 2. Select the target (what we want to predict)
y_raw = df_cleaned['ProductCategory']

# 3. Initialize the LabelEncoder to turn text into numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)

# Print the results to see how categories were mapped
print("Category Mapping:")
for index, label in enumerate(label_encoder.classes_):
    print(f"{index}: {label}")

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y_encoded.shape}")

Category Mapping:
0: Beauty
1: Books
2: Clothing
3: Electronics
4: Home & Garden
5: Sports

Features shape: (8124, 3)
Target shape: (8124,)


**Step 3: Splitting the Data**

We divide the data to ensure we can evaluate the model on information it hasn't seen before.

In [119]:
from sklearn.model_selection import train_test_split

# Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")

Training set size: 6499 rows
Test set size: 1625 rows


**Step 4: Scaling the Features**

Scaling ensures that features with larger numerical ranges (like TotalAmount) don't dominate features with smaller ranges (like Quantity).

In [120]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")
print(f"Scaled training data (first row):\n{X_train_scaled[0]}")

Features scaled successfully.
Scaled training data (first row):
[-0.35527394  0.35668696  0.15319385]


**Step 5: Training and Evaluating the Model**

We will use Logistic Regression to train our model on the scaled data and then measure its accuracy.

In [121]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize the model
model = LogisticRegression(random_state=42)

# Train the model using the scaled training data
model.fit(X_train_scaled, y_train)

# Make predictions on the scaled test set
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline Logistic Regression Accuracy: {accuracy:.4f}")

Baseline Logistic Regression Accuracy: 0.4332


**Step 6: Training a Random Forest Model**

We will now use a Random Forest algorithm to see if it can achieve a higher accuracy than our baseline model.

In [122]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and train a Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test_scaled)

# Calculate new accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Model Accuracy: {accuracy_rf:.4f}")
print(f"Improvement over baseline: {(accuracy_rf - accuracy):.4f}")

Random Forest Model Accuracy: 0.4129
Improvement over baseline: -0.0203


**Step 7: Hyperparameter Tuning**

We will use Grid Search to find the best configuration for our Random Forest to improve its accuracy.

In [123]:
from sklearn.model_selection import GridSearchCV

# Define the settings we want to test
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20]
}

# Initialize the search
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)

# Run the search on our training data
grid_search.fit(X_train_scaled, y_train)

# Get the best version of the model
best_rf_model = grid_search.best_estimator_

# Test the best model
y_pred_best_rf = best_rf_model.predict(X_test_scaled)
accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Tuned Random Forest Accuracy: {accuracy_best_rf:.4f}")
print(f"Improvement over baseline: {(accuracy_best_rf - accuracy):.4f}")

Best Parameters: {'max_depth': 10, 'n_estimators': 100}
Tuned Random Forest Accuracy: 0.5175
Improvement over baseline: 0.0843
